# Notebook 9: extensions on the rheumatic and autoimmune disease dataset

This notebook has two parts. Both use the same data, fitness (XGBoost + HistGradientBoosting with SMOTE, 3-fold CV on a 4,000-patient subsample), test models (200 trees), population (12) and budget (252 evaluations) as Notebook 6.

**Part A: the original split** (random_state = 42, seeds 0–9). The fitness cache from Notebooks 1, 4 and 6 is reused, so this part is fast.
- GA+SI and HO+SI (the intervention layer around GA and HO).
- RL-DOHO with the Q-learning controller, at the same budget and at 3× the budget.
- RL-DOHO with κ = 1, 3 and 4.

**Part B: repeated train/test splits.** This answers "the clinical result rests on one split".
- Four new stratified 80/20 splits (random_state = 1, 2, 3, 4), with seeds 0–4 on each.
- On each new split, the imputation, the scaling and the 4,000-patient fitness subsample are all refitted on that split's training data.
- Methods: RL-DOHO, DO, HO, DOHO, GA, BPSO, BGWO, GA+SI, HO+SI, LASSO, mRMR, ReliefF and all features.
- These are combined with seeds 0–4 of the original split, giving 5 splits × 5 seeds = 25 paired runs per method.

**Runtime:**
- Part A takes about 1–2 hours.
- Part B needs a fresh fitness cache for every split (about 2 s per new subset on Colab), so expect about 2 hours per split, roughly 8 hours for four splits.
- Every run is saved as soon as it finishes, so you can stop and resume at any time.
- By default Part B runs two new splits (1 and 2), giving 3 splits × 5 seeds = 15 paired runs per method. Add 3 and 4 if you have time.

**If the session keeps dropping:**
- Nothing finished is lost. Results and the fitness cache are saved every 25 new evaluations, and saving can no longer corrupt a file. Rerun all cells from the top and the notebook continues where it stopped; finished runs are skipped in seconds.
- **Run splits in parallel.** Open this notebook in a second Colab session, set `EXTRA_SPLITS = [2]` in one and `[1]` in the other, and run both. Each split writes its own files.
- **Faster still: run it on your own PC.** A 6-core Ryzen is roughly 2–3× faster than free Colab's 2 cores and never disconnects. Copy the data file and the folders `rl_doho_results`, `rl_doho_results_v2_rheum` and `rl_doho_results_v3_rheum` from Drive into one folder, put this notebook in the same folder, and run it with Jupyter or VS Code.

### Fixed plan for the extension experiments (written before any of these runs; every outcome is reported)

**Why these runs:** they answer the main objections a reviewer would raise about the v2 results.
1. *"Is the gain from the controller or from the interventions?"* A second learning controller (tabular **Q-learning**) is added next to UCB1 and random arm choice.
2. *"Would the interventions help any optimiser?"* The same intervention layer (stagnation trigger, PERTURB, RESTART, tabu memory, tie-break, UCB1 with the gain reward) is wrapped around **GA** and **HO**. These are called **GA+SI** and **HO+SI** (SI = stagnation-triggered interventions). Their arms are PERTURB, RESTART and one iteration of the base optimiser.
3. *"How sensitive is RL-DOHO to κ?"* κ ∈ {1, 2, 3, 4}; κ = 2 is the main setting and is reused.
4. *"What does it cost?"* Wall-clock time is split into the algorithm's own overhead and the time spent in the fitness function. The number of distinct subsets each run evaluates is also recorded.

**Q-learning controller (settings fixed now, not tuned)**
- State = budget phase (first, middle or last third of the budget) × whether the previous intervention improved the best subset. That gives 6 states.
- Actions = the four arms. Reward = the same gain reward as UCB1.
- One-step Q-learning: learning rate 0.3, discount 0.5, ε-greedy with ε = 0.2, Q starts at 0.
- The Q-table starts empty in every run, like UCB1.

**Hypotheses (fixed in advance)**
- **H5:** At 3× the budget, Q-learning vs random arm choice. No direction predicted; pooled two-sided sign test on fitness.
- **H6:** GA+SI reaches lower fitness than GA, and HO+SI lower than HO. Pooled two-sided sign tests, Holm-corrected over the two comparisons.
- **H7:** κ has no effect on RL-DOHO's fitness. Friedman test over κ ∈ {1, 2, 3, 4} on the pooled runs, with per-run ranks reported.
- All other comparisons (RL-DOHO against every method on the new datasets, Q-learning vs UCB1 at the same budget, GA+SI vs RL-DOHO) are reported with the same tests as Notebooks 6–7. No direction is predicted for them.

A difference below 1e-12 counts as a tie, as before.

**Added for this notebook (fixed in advance):**
- **H8:** Over the 25 paired runs of Part B, RL-DOHO vs each other method. No direction predicted. Pooled two-sided sign tests, Holm-corrected per metric, plus Friedman tests with splits × seeds as blocks.
- RL-DOHO's rank among the 13 methods is reported for every split separately, so it is visible whether the original split was typical.

In [10]:
# ================= 1. SETUP =================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import subprocess, sys
try:
    import xgboost, imblearn
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost", "imbalanced-learn"], check=True)

import os, time, pickle, warnings, math
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# ================= 2. CONFIG =================
ROOT      = '/content/drive/MyDrive' if IN_COLAB else '.'   # running on your own PC: put the data file and the result folders next to this notebook
DATA_PATH = f"{ROOT}/Rheumatic and Autoimmune Disease Dataset.xlsx"
PREV_DIR  = f"{ROOT}/rl_doho_results"              # Notebooks 1 and 4: fitness cache, GA/BPSO/BGWO/filter runs
V2_DIR    = f"{ROOT}/rl_doho_results_v2_rheum"     # Notebook 6: runs, ablation, long budget, cache
OUT_DIR   = f"{ROOT}/rl_doho_results_v3_rheum"     # results of this notebook
SEEDS       = list(range(10))                      # Part A
SPLIT_SEEDS = list(range(5))                       # Part B: seeds per split
ORIG_SPLIT  = 42
EXTRA_SPLITS = [1, 2]                              # Part B: new train/test splits. Each split is independent and saves to its own
                                                   # files, so you can run split 1 in one session and split 2 in another at the same time
POP_SIZE  = 12
BUDGET    = POP_SIZE * 21                          # 252 evaluations
LONG_BUDGET = 3 * BUDGET
INIT      = "dense"
CV_FOLDS, FIT_TREES, EVAL_TREES, USE_SMOTE = 3, 60, 200, True
ALPHA, BETA = 0.99, 0.01
FITNESS_SUBSAMPLE = 4000
VERBOSE   = False
RUN = dict(part_a=True, part_a_long=True, part_b=True)   # finished parts are skipped automatically

FAST = False                                       # True: tiny settings, only to check that the notebook runs
if FAST:
    SEEDS, SPLIT_SEEDS, EXTRA_SPLITS = [0], [0], [1]
    BUDGET, LONG_BUDGET, FITNESS_SUBSAMPLE, EVAL_TREES = 48, 96, 1000, 30

os.makedirs(OUT_DIR, exist_ok=True)
FAMILY   = ["RL-DOHO", "DO", "HO", "DOHO"]
WRAPPERS = FAMILY + ["GA", "BPSO", "BGWO"]
ADDONS   = ["GA+SI", "HO+SI"]
FILTERS  = ["LASSO", "mRMR", "ReliefF", "All features"]
METHODS  = WRAPPERS + ADDONS + FILTERS
def save_pkl(obj, path):
    # write to a temporary file first, then swap it in: an interruption mid-save can no longer corrupt the results
    tmp = path + ".tmp"
    with open(tmp, "wb") as f: pickle.dump(obj, f)
    os.replace(tmp, path)
def load_pkl(path, default=None):
    for p in (path, path + ".tmp"):
        if os.path.exists(p):
            try:
                with open(p, "rb") as f: return pickle.load(f)
            except Exception as e:
                print(f"WARNING: could not read {p} ({e}); trying the backup")
    return {} if default is None else default
print("Output folder:", OUT_DIR, "| budget:", BUDGET, "| extra splits:", EXTRA_SPLITS)

Output folder: /content/drive/MyDrive/rl_doho_results_v3_rheum | budget: 252 | extra splits: [1, 2]


## 3. Data, with preprocessing refitted for every train/test split

In [12]:
df = pd.read_excel(DATA_PATH)
numeric_cols = ['Age', 'ESR', 'CRP', 'RF', 'Anti-CCP', 'C3', 'C4']
binary_cols  = ['Gender', 'HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm']
FEATURES = numeric_cols + binary_cols
N_FEATURES = len(FEATURES)
X_all = df[FEATURES].copy()
for c in binary_cols:
    X_all[c] = X_all[c].astype(str).str.strip().str.lower().map({'male': 1, 'female': 0, 'positive': 1, 'negative': 0, 'yes': 1, 'no': 0})
y_all = LabelEncoder().fit_transform(df['Disease'].astype(str))

SPLIT = {}
def make_split(sp):
    # sp = 42 reproduces Notebooks 1-6 exactly (split random_state 42, fitness subsample random_state 0)
    Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=sp)
    Xtr, Xte = Xtr.reset_index(drop=True), Xte.reset_index(drop=True)
    num_imp = SimpleImputer(strategy='mean').fit(Xtr[numeric_cols]); bin_imp = SimpleImputer(strategy='most_frequent').fit(Xtr[binary_cols])
    scaler = StandardScaler().fit(num_imp.transform(Xtr[numeric_cols]))
    def prep(Xp):
        out = Xp.copy(); out[numeric_cols] = scaler.transform(num_imp.transform(Xp[numeric_cols])); out[binary_cols] = bin_imp.transform(Xp[binary_cols])
        return out.astype(float)
    Xtr, Xte = prep(Xtr), prep(Xte)
    Xfit, _, yfit, _ = train_test_split(Xtr, ytr, train_size=FITNESS_SUBSAMPLE, stratify=ytr, random_state=0 if sp == ORIG_SPLIT else sp)
    return dict(X_train=Xtr, X_test=Xte, y_train=ytr, y_test=yte, X_fit=Xfit.reset_index(drop=True), y_fit=yfit)
for sp in [ORIG_SPLIT] + EXTRA_SPLITS:
    SPLIT[sp] = make_split(sp)
print("Rows:", len(df), "| splits prepared:", list(SPLIT), "| fitness set per split:", SPLIT[ORIG_SPLIT]["X_fit"].shape)

Rows: 12085 | splits prepared: [42, 1, 2] | fitness set per split: (4000, 14)


## 4. Fitness and test evaluation (identical to Notebook 6), with one cache per split

In [13]:
CV = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
def make_models(n_trees):
    xgb = XGBClassifier(n_estimators=n_trees, max_depth=4, learning_rate=0.1, random_state=42, tree_method='hist', verbosity=0, n_jobs=-1)
    hgb = HistGradientBoostingClassifier(max_iter=n_trees, max_depth=4, random_state=42)
    if USE_SMOTE:
        xgb = ImbPipeline([('smote', SMOTE(random_state=42)), ('clf', xgb)])
        hgb = ImbPipeline([('smote', SMOTE(random_state=42)), ('clf', hgb)])
    return xgb, hgb

def mask_str(m): return ''.join(map(str, np.asarray(m).astype(int)))
ORIG_CACHE = f"fitness_cache_{FIT_TREES}t_{CV_FOLDS}f_{int(USE_SMOTE)}s_{FITNESS_SUBSAMPLE}.pkl"   # name used by Notebooks 1-6
def cache_file(sp): return ORIG_CACHE if sp == ORIG_SPLIT else f"fitness_cache_split{sp}_{FITNESS_SUBSAMPLE}.pkl"
CACHES, TEST_CACHES, NEW_EVAL = {}, {}, {}
def load_cache(sp):
    if sp in CACHES: return
    c = {}
    for d in (PREV_DIR, V2_DIR, OUT_DIR):
        p = f"{d}/{cache_file(sp)}"
        c.update(load_pkl(p))
    for k, v in c.items():
        n = k.count("1"); v["fit"] = 1.0 if n == 0 else ALPHA * (1 - v["acc"]) + BETA * n / len(k)
    CACHES[sp] = c; TEST_CACHES[sp] = {}
    tp = f"{OUT_DIR}/test_cache_split{sp}_{EVAL_TREES}.pkl"
    TEST_CACHES[sp] = load_pkl(tp)
def save_cache(sp):
    save_pkl(CACHES[sp], f"{OUT_DIR}/{cache_file(sp)}")
    save_pkl(TEST_CACHES[sp], f"{OUT_DIR}/test_cache_split{sp}_{EVAL_TREES}.pkl")

CUR = {"sp": ORIG_SPLIT}
def use_split(sp):
    load_cache(sp); CUR["sp"] = sp

def fitness_function(mask):
    sp = CUR["sp"]; cache = CACHES[sp]; key = mask_str(mask)
    if key in cache: return cache[key]["fit"]
    sel = np.flatnonzero(mask)
    if len(sel) == 0:
        cache[key] = {"fit": 1.0, "acc": 0.0}; return 1.0
    S = SPLIT[sp]; Xs = S["X_fit"].iloc[:, sel]; xgb, hgb = make_models(FIT_TREES); t0 = time.perf_counter()
    acc = (cross_val_score(xgb, Xs, S["y_fit"], cv=CV).mean() + cross_val_score(hgb, Xs, S["y_fit"], cv=CV).mean()) / 2
    f = ALPHA * (1 - acc) + BETA * len(sel) / len(mask)
    cache[key] = {"fit": f, "acc": acc}
    c = NEW_EVAL.setdefault(sp, [0, 0.0]); c[0] += 1; c[1] += time.perf_counter() - t0
    if c[0] % 10 == 0:
        print(".", end="", flush=True)
        if c[0] % 25 == 0: save_cache(sp)          # at most about a minute of work is lost if the session drops
    return f

def test_metrics(mask):
    sp = CUR["sp"]; key = mask_str(mask); tc = TEST_CACHES[sp]
    if key not in tc:
        S = SPLIT[sp]; idx = np.flatnonzero(mask); preds = []
        for m in make_models(EVAL_TREES):
            m.fit(S["X_train"].iloc[:, idx], S["y_train"]); preds.append(m.predict(S["X_test"].iloc[:, idx]))
        yt = S["y_test"]
        tc[key] = {"test_acc":  float(np.mean([accuracy_score(yt, p) for p in preds])),
                   "test_prec": float(np.mean([precision_score(yt, p, average="macro", zero_division=0) for p in preds])),
                   "test_rec":  float(np.mean([recall_score(yt, p, average="macro", zero_division=0) for p in preds])),
                   "test_f1":   float(np.mean([f1_score(yt, p, average="macro") for p in preds]))}
    return tc[key]
use_split(ORIG_SPLIT)
print(f"Original-split cache: {len(CACHES[ORIG_SPLIT])} subsets already evaluated")

Original-split cache: 10273 subsets already evaluated


## 5. Algorithms: the Notebook 6 code plus the Q-learning controller, the GA+SI / HO+SI add-on, fitness timing and the filters

In [14]:
# ===================== RL-DOHO v2: shared algorithm code =====================
# Every method gets the same number of FITNESS EVALUATIONS (not iterations), because the
# published HO evaluates about 3N candidates per iteration while DO evaluates N.
# Encoding: continuous position x in [LB, UB]^D, feature j selected if sigmoid(x_j) > 0.5 (i.e. x_j > 0).
import math, time
import numpy as np, pandas as pd

LB, UB = -3.0, 3.0

class BudgetExhausted(Exception):
    pass

def binarize(x):
    return (np.asarray(x) > 0).astype(np.int8)          # identical to sigmoid(x) > 0.5

def mkey(mask):
    return np.packbits(np.asarray(mask, dtype=np.uint8)).tobytes()

class Problem:
    """Wraps a (cached, deterministic) fitness function and enforces the evaluation budget.
    Also records the time spent inside the fitness function (fsec) and the number of distinct
    subsets evaluated (len(seen)), so the algorithm's own overhead can be separated from fitness cost."""
    def __init__(self, fitness_fn, D, budget):
        self.fitness_fn, self.D, self.budget, self.calls = fitness_fn, D, budget, 0
        self.fsec, self.seen = 0.0, set()
    def left(self):
        return self.budget - self.calls
    def f(self, mask):
        if self.calls >= self.budget:
            raise BudgetExhausted
        self.calls += 1
        m = np.asarray(mask, dtype=np.int8); self.seen.add(mkey(m))
        t0 = time.perf_counter(); v = float(self.fitness_fn(m)); self.fsec += time.perf_counter() - t0
        return v
    def fx(self, x):
        return self.f(binarize(x))

# ------------------------------------------------------------------ initialisation
def init_population(rng, N, D, mode="dense"):
    """dense : x ~ U(-1, 1)  -> each feature selected with probability 0.5 (as in Notebooks 1-5).
       sparse: agent i keeps a fraction p_i ~ log-uniform[min(10/D, 0.1), 0.1] of the features
               (used when D > 100: ten features up to 10% of them)."""
    if mode == "dense":
        return rng.uniform(-1, 1, (N, D))
    lo, hi = math.log(min(10.0 / D, 0.1)), math.log(0.1)
    X = np.empty((N, D))
    for i in range(N):
        p = math.exp(rng.uniform(lo, hi))
        sel = rng.random(D) < p
        if not sel.any():
            sel[rng.integers(D)] = True
        X[i] = np.where(sel, rng.uniform(0.1, 1.0, D), -rng.uniform(0.1, 1.0, D))
    return X

def random_position(rng, D, mode):
    return init_population(rng, 1, D, mode)[0]

# ------------------------------------------------------------------ Levy flight (Mantegna)
def levy(rng, shape, beta=1.5):
    num = math.gamma(1 + beta) * math.sin(math.pi * beta / 2)
    den = math.gamma((1 + beta) / 2) * beta * 2 ** ((beta - 1) / 2)
    sigma = (num / den) ** (1 / beta)
    u = rng.normal(0, sigma, shape); v = rng.normal(0, 1, shape)
    return u / np.abs(v) ** (1 / beta)

def lognpdf(x):                                          # lognormal pdf, mu = 0, sigma = 1
    x = np.maximum(x, 1e-300)
    return np.exp(-np.log(x) ** 2 / 2) / (x * math.sqrt(2 * math.pi))

# ------------------------------------------------------------------ Dandelion Optimizer (Zhao et al., 2022)
def do_move(rng, X, elite, t, T):
    """One full DO iteration: rising (Eqs. 2-8), descending (Eqs. 10-11), landing (Eqs. 12-14).
    Returns the new population; the caller evaluates it (N evaluations)."""
    N, D = X.shape
    T = max(T, 2); t = min(max(t, 1), T)
    alpha = rng.random() * ((t / T) ** 2 - 2 * t / T + 1)                     # Eq. 4: adaptive step
    a = 1.0 / (T ** 2 - 2 * T + 1); b = -2 * a; c = 1 - a - b
    k = 1 - rng.random() * (c + a * t ** 2 + b * t)                          # Eq. 8 (k with q)
    # --- rising stage
    if rng.standard_normal() < 1.5:                                          # clear weather
        lamb = np.abs(rng.standard_normal((N, D)))
        theta = (2 * rng.random(N) - 1) * math.pi
        r = 1 / np.exp(theta); vx = r * np.cos(theta); vy = r * np.sin(theta)
        Xs = rng.uniform(LB, UB, (N, D))                                     # Eq. 3: random position
        X1 = X + alpha * (vx * vy)[:, None] * lognpdf(lamb) * (Xs - X)       # Eq. 2
    else:                                                                    # rainy weather
        X1 = X * k                                                           # Eq. 7
    X1 = np.clip(X1, LB, UB)
    # --- descending stage (Brownian motion around the population mean)
    beta = rng.standard_normal((N, D))
    Xm = X1.mean(axis=0)                                                     # Eq. 11
    X2 = np.clip(X1 - beta * alpha * (Xm - beta * alpha * X1), LB, UB)      # Eq. 10
    # --- landing stage (Levy flight around the elite)
    step = 0.01 * levy(rng, (N, D))                                          # Eq. 13, s = 0.01, beta = 1.5
    delta = 2 * t / T                                                        # Eq. 14
    X3 = elite + step * alpha * (elite - X2 * delta)                         # Eq. 12
    return np.clip(X3, LB, UB)

# ------------------------------------------------------------------ Hippopotamus Optimization (Amiri et al., 2024)
def ho_iteration(rng, X, F, dominant, t, T, prob, on_eval):
    """One full HO iteration with greedy selection, updating X, F in place.
    Phase 1 (first half): male + female/immature candidates (Eqs. 3-9)      -> 2 evaluations per agent
    Phase 2 (second half): predator + defence candidate (Eqs. 10-16)        -> 2 evaluations per agent
    Phase 3 (all agents): escape within shrinking local bounds (Eqs. 17-19) -> 1 evaluation per agent
    on_eval(x, f) is called for every evaluated candidate that is a hippopotamus position."""
    N, D = X.shape
    half = N // 2
    T = max(T, 1); t = min(max(t, 1), T)
    for i in range(half):
        I1, I2 = rng.integers(1, 3, 2); rho = rng.integers(0, 2, 2)
        g = int(rng.integers(1, N + 1)); grp = rng.choice(N, g, replace=False)
        MG = X[grp].mean(axis=0)
        h = [I2 * rng.random(D) + (1 - rho[0]), 2 * rng.random(D) - 1, rng.random(D),
             I1 * rng.random(D) + (1 - rho[1]), rng.random()]                  # Eq. 4
        h1, h2 = h[rng.integers(5)], h[rng.integers(5)]
        x_m = np.clip(X[i] + rng.random() * (dominant - I1 * X[i]), LB, UB)     # Eq. 3 (male)
        if math.exp(-t / T) > 0.6:                                            # Eq. 5
            x_f = X[i] + h1 * (dominant - I2 * MG)                            # Eq. 6
        elif rng.random() > 0.5:
            x_f = X[i] + h2 * (MG - dominant)                                 # Eq. 7
        else:
            x_f = rng.uniform(LB, UB, D)
        x_f = np.clip(x_f, LB, UB)
        for x in (x_m, x_f):                                                  # Eqs. 8-9: greedy
            f = prob.fx(x); on_eval(x, f)
            if f < F[i]: X[i], F[i] = x, f
    for i in range(half, N):
        pred = rng.uniform(LB, UB, D)                                         # Eq. 10
        f_pred = prob.fx(pred)
        dist = np.abs(pred - X[i]) + 1e-12                                    # Eq. 11
        b_, c_, d_ = rng.uniform(2, 4), rng.uniform(1, 1.5), rng.uniform(2, 3)
        l_ = rng.uniform(-2 * math.pi, 2 * math.pi)                           # 2*pi*g with g in [-1, 1]
        RL = 0.05 * levy(rng, D)                                              # Eq. 13
        if F[i] > f_pred:
            x = RL * pred + (b_ / (c_ - d_ * math.cos(l_))) * (1 / dist)      # Eq. 12, case 1
        else:
            x = RL * pred + (b_ / (c_ - d_ * math.cos(l_))) * (1 / (2 * dist + rng.random(D)))
        x = np.clip(x, LB, UB)
        f = prob.fx(x); on_eval(x, f)
        if f < F[i]: X[i], F[i] = x, f                                        # Eqs. 15-16
    lo, hi = LB / t, UB / t                                                   # Eq. 17: local bounds
    for i in range(N):
        s = [2 * rng.random(D) - 1, rng.random(), rng.standard_normal()][rng.integers(3)]   # Eq. 19
        x = np.clip(X[i] + rng.random() * (lo + s * (hi - lo)), LB, UB)      # Eq. 18
        f = prob.fx(x); on_eval(x, f)
        if f < F[i]: X[i], F[i] = x, f                                        # Eq. 20

# ------------------------------------------------------------------ run bookkeeping
class Tracker:
    """Tracks the best subset found and logs one row per iteration."""
    def __init__(self, name, prob, verbose=False, tie_eps=0.0):
        self.name, self.prob, self.verbose, self.tie_eps = name, prob, verbose, tie_eps
        self.best_f, self.best_x, self.best_m = np.inf, None, None
        self.rows, self.t0 = [], time.time()
    def better(self, f, nf):
        if self.best_m is None: return True
        bf, bn = self.best_f, int(self.best_m.sum())
        e = self.tie_eps
        if e <= 0: return f < bf
        return f < bf - e or (abs(f - bf) <= e and (nf < bn or (nf == bn and f < bf)))
    def offer(self, x, f, mask=None):
        m = binarize(x) if mask is None else mask
        if self.better(f, int(m.sum())):
            self.best_f, self.best_x, self.best_m = f, (None if x is None else np.array(x, float)), m.copy()
            return True
        return False
    def log(self, it, action, F=None, reward=None):
        row = dict(iter=it, evals=self.prob.calls, action=action, best=self.best_f,
                   n_feats=int(self.best_m.sum()), pop_mean=(float(np.mean(F)) if F is not None else np.nan),
                   reward=reward, sec=time.time() - self.t0)
        self.rows.append(row)
        if self.verbose:
            rw = "" if reward is None else f"  r={reward:.3f}"
            print(f"  {self.name:<16} it {it:>3}  evals {self.prob.calls:>5}/{self.prob.budget}  {action:<10}"
                  f" best {self.best_f:.5f}  feats {row['n_feats']:>5}{rw}")
    def result(self):
        return {"mask": self.best_m, "fit": self.best_f, "hist": pd.DataFrame(self.rows)}

def _start(prob, rng, N, init, tr):
    X = init_population(rng, N, prob.D, init)
    F = np.array([prob.fx(x) for x in X])
    for x, f in zip(X, F): tr.offer(x, f)
    tr.log(0, "init", F)
    return X, F

# ------------------------------------------------------------------ DO, HO, DOHO (published update rules)
def run_do(prob, N, seed, init="dense", verbose=False):
    rng = np.random.default_rng(seed); tr = Tracker("DO", prob, verbose)
    T = max(2, (prob.budget - N) // N)
    try:
        X, F = _start(prob, rng, N, init, tr)
        for t in range(1, T + 1):
            X = do_move(rng, X, tr.best_x, t, T)
            F = np.array([prob.fx(x) for x in X])
            for x, f in zip(X, F): tr.offer(x, f)
            tr.log(t, "DO", F)
    except BudgetExhausted:
        pass
    return tr.result()

def run_ho(prob, N, seed, init="dense", verbose=False):
    rng = np.random.default_rng(seed); tr = Tracker("HO", prob, verbose)
    T = max(1, math.ceil((prob.budget - N) / (3 * N)))
    try:
        X, F = _start(prob, rng, N, init, tr)
        for t in range(1, T + 1):
            dom = X[np.argmin(F)].copy()
            ho_iteration(rng, X, F, dom, t, T, prob, tr.offer)
            tr.log(t, "HO", F)
    except BudgetExhausted:
        if tr.rows and tr.rows[-1]["evals"] != prob.calls: tr.log(len(tr.rows), "HO (partial)", F)
    return tr.result()

def run_doho(prob, N, seed, init="dense", verbose=False):
    """Fixed switch: published DO for the first half of the budget, then published HO."""
    rng = np.random.default_rng(seed); tr = Tracker("DOHO", prob, verbose)
    half_budget = prob.budget // 2
    T1 = max(2, (half_budget - N) // N); T2 = max(1, math.ceil((prob.budget - half_budget) / (3 * N)))
    it = 0
    try:
        X, F = _start(prob, rng, N, init, tr)
        for t in range(1, T1 + 1):
            if prob.calls + N > half_budget: break
            X = do_move(rng, X, tr.best_x, t, T1)
            F = np.array([prob.fx(x) for x in X])
            for x, f in zip(X, F): tr.offer(x, f)
            it += 1; tr.log(it, "DO", F)
        for t in range(1, T2 + 1):
            ho_iteration(rng, X, F, X[np.argmin(F)].copy(), t, T2, prob, tr.offer)
            it += 1; tr.log(it, "HO", F)
    except BudgetExhausted:
        pass
    return tr.result()

# ------------------------------------------------------------------ controllers
class UCB1:
    def __init__(self, n_arms, seed, c=1.0):
        self.rng = np.random.default_rng(seed + 7919); self.c = c
        self.n = np.zeros(n_arms); self.v = np.zeros(n_arms)
    def choose(self, state=None):
        untried = np.flatnonzero(self.n == 0)
        if len(untried): return int(self.rng.choice(untried)), "i"
        ucb = self.v + self.c * np.sqrt(2 * np.log(self.n.sum()) / self.n)
        return int(self.rng.choice(np.flatnonzero(np.isclose(ucb, ucb.max())))), "u"
    def update(self, a, r):
        self.n[a] += 1; self.v[a] += (r - self.v[a]) / self.n[a]
    def finish(self):
        pass

class RandomArm(UCB1):
    def choose(self, state=None): return int(self.rng.integers(len(self.n))), "r"

class FixedArm(UCB1):
    def __init__(self, n_arms, seed, arm=0):
        super().__init__(n_arms, seed); self.arm = arm
    def choose(self, state=None): return self.arm, "f"

N_STATES = 6
def controller_state(calls, budget, last_gain):
    """State for the Q-learning controller: budget phase (first, second, last third) x whether the
    previous intervention improved the best subset (0/1) -> 6 states."""
    phase = min(2, int(3 * calls / budget))
    return 2 * phase + (1 if last_gain > 0 else 0)

class QLearn(UCB1):
    """Tabular one-step Q-learning over the arms. Epsilon-greedy (eps = 0.2), learning rate 0.3,
    discount 0.5, Q initialised at 0. The next state is the state at the next decision; the last
    decision of a run is updated without bootstrapping. Settings fixed in advance, not tuned."""
    def __init__(self, n_arms, seed, n_states=N_STATES, lr=0.3, gamma=0.5, eps=0.2):
        super().__init__(n_arms, seed)
        self.Q = np.zeros((n_states, n_arms)); self.lr, self.gamma, self.eps = lr, gamma, eps
        self.pending, self.s = None, 0
    def choose(self, state=0):
        if self.pending is not None:
            s, a, r = self.pending
            self.Q[s, a] += self.lr * (r + self.gamma * self.Q[state].max() - self.Q[s, a]); self.pending = None
        self.s = state
        if self.rng.random() < self.eps:
            return int(self.rng.integers(self.Q.shape[1])), "e"
        q = self.Q[state]
        return int(self.rng.choice(np.flatnonzero(np.isclose(q, q.max())))), "q"
    def update(self, a, r):
        self.n[a] += 1; self.pending = (self.s, a, r)
    def finish(self):
        if self.pending is not None:
            s, a, r = self.pending; self.Q[s, a] += self.lr * (r - self.Q[s, a]); self.pending = None

CONTROLLERS = {"ucb": UCB1, "random": RandomArm, "qlearn": QLearn}

ARMS = ("PERTURB", "RESTART", "DO", "HO")

# ------------------------------------------------------------------ RL-DOHO v2
def run_rldoho(prob, N, seed, init="dense", verbose=False, controller="ucb", reward="gain",
               kappa=2, tie_eps=1e-3, k_max=3, tabu_tries=20, intervene=True, name="RL-DOHO"):
    """DO (published) while the best improves; after `kappa` iterations without an accepted improvement,
    a controller picks one arm: PERTURB / RESTART (worse half, tabu) or one published DO / HO iteration.
    reward="gain": 0.8 * min(1, g / g_max) + 0.2 * share, where g = fitness gain of the best per evaluation
                   spent (parsimony-only moves give g = 0) and g_max = largest g seen so far in the run.
    reward="old" : 1 if the best changed, otherwise 0.2 * share (Notebooks 1-5)."""
    rng = np.random.default_rng(seed); D = prob.D
    tr = Tracker(name, prob, verbose, tie_eps=tie_eps)
    ctrl = CONTROLLERS.get(controller)
    ctrl = ctrl(len(ARMS), seed) if ctrl else FixedArm(len(ARMS), seed, arm=ARMS.index(controller))
    T_do = max(2, (prob.budget - N) // N); T_ho = max(1, math.ceil((prob.budget - N) / (3 * N)))
    def sched(T):                     # DO / HO schedules follow the share of the budget already spent
        return min(T, max(1, math.ceil(T * prob.calls / prob.budget)))
    tabu = set(); g_max = 0.0; stag = 0; it = 0; last_gain = 0.0
    arm_log = []
    def offer(x, f):
        tabu.add(mkey(binarize(x)))
        return tr.offer(x, f)
    def tabu_draw(make):
        for _ in range(tabu_tries):
            x = np.clip(make(), LB, UB)
            if mkey(binarize(x)) not in tabu: break
        return x
    try:
        X = init_population(rng, N, D, init)
        F = np.array([prob.fx(x) for x in X])
        for x, f in zip(X, F): offer(x, f)
        tr.log(0, "init", F)
        while prob.left() > 0:
            it += 1
            prev_best, prev_feats, prev_med, calls0 = tr.best_f, int(tr.best_m.sum()), float(np.median(F)), prob.calls
            changed_before = (tr.best_f, int(tr.best_m.sum()))
            if not intervene or stag < kappa:
                arm, how = None, None
                X = do_move(rng, X, tr.best_x, sched(T_do), T_do)
                F = np.array([prob.fx(x) for x in X])
                for x, f in zip(X, F): offer(x, f)
                moved = np.arange(N)
            else:
                arm, how = ctrl.choose(controller_state(prob.calls, prob.budget, last_gain)); a = ARMS[arm]
                if a == "DO":
                    X = do_move(rng, X, tr.best_x, sched(T_do), T_do)
                    F = np.array([prob.fx(x) for x in X])
                    for x, f in zip(X, F): offer(x, f)
                    moved = np.arange(N)
                elif a == "HO":
                    F_before = F.copy()
                    ho_iteration(rng, X, F, X[np.argmin(F)].copy(), sched(T_ho), T_ho, prob, offer)
                    moved = np.flatnonzero(F != F_before)
                else:
                    worst = np.argsort(F)[N // 2:]
                    for i in worst:
                        if a == "PERTURB":
                            def make():
                                x = tr.best_x.copy(); k = int(rng.integers(1, k_max + 1))
                                j = rng.choice(D, k, replace=False)
                                x[j] = -np.sign(x[j] + 1e-12) * rng.uniform(0.5, 2.0, k)
                                return x
                        else:
                            make = lambda: random_position(rng, D, init)
                        X[i] = tabu_draw(make)
                        F[i] = prob.fx(X[i]); offer(X[i], F[i])
                    moved = worst
            cost = max(1, prob.calls - calls0)
            changed = (tr.best_f, int(tr.best_m.sum())) != changed_before
            stag = 0 if changed else stag + 1
            g = max(0.0, prev_best - tr.best_f) / cost
            g_max = max(g_max, g)
            if arm is None:
                tr.log(it, "explore", F)
            else:
                share = float(np.mean(F[moved] < prev_med)) if len(moved) else 0.0
                if reward == "gain":
                    r = 0.8 * (min(1.0, g / g_max) if g_max > 0 else 0.0) + 0.2 * share
                else:
                    r = 1.0 if changed else 0.2 * share
                ctrl.update(arm, r); last_gain = prev_best - tr.best_f
                arm_log.append((it, ARMS[arm], how, r, prev_best - tr.best_f, cost))
                tr.log(it, f"{ARMS[arm]}({how})", F, reward=r)
    except BudgetExhausted:
        pass
    ctrl.finish()
    res = tr.result()
    res["arms"] = pd.DataFrame(arm_log, columns=["iter", "arm", "how", "reward", "gain", "cost"])
    return res

# ------------------------------------------------------------------ GA, BPSO, BGWO (same budget and initial population)
def run_ga(prob, N, seed, init="dense", verbose=False, pc=0.9, tour=3):
    rng = np.random.default_rng(seed); tr = Tracker("GA", prob, verbose); D = prob.D; pm = 1.0 / D
    try:
        X, F = _start(prob, rng, N, init, tr)
        M = binarize(X); it = 0
        while True:
            def pick():
                idx = rng.choice(N, tour, replace=False); return M[idx[np.argmin(F[idx])]]
            new = [tr.best_m.copy()]
            while len(new) < N:
                p1, p2 = pick(), pick()
                if rng.random() < pc:
                    cm = rng.random(D) < 0.5; kids = [np.where(cm, p1, p2), np.where(cm, p2, p1)]
                else:
                    kids = [p1.copy(), p2.copy()]
                for c in kids:
                    c = np.where(rng.random(D) < pm, 1 - c, c).astype(np.int8)
                    if len(new) < N: new.append(c)
            M = np.array(new, dtype=np.int8)
            F = np.empty(N)
            for i in range(N):
                F[i] = prob.f(M[i]); tr.offer(None, F[i], mask=M[i])
            it += 1; tr.log(it, "gen", F)
    except BudgetExhausted:
        pass
    return tr.result()

def run_bpso(prob, N, seed, init="dense", verbose=False, c1=2.0, c2=2.0, vmax=6.0):
    rng = np.random.default_rng(seed); tr = Tracker("BPSO", prob, verbose); D = prob.D
    T = max(2, (prob.budget - N) // N)
    try:
        X0, F = _start(prob, rng, N, init, tr)
        X = binarize(X0); V = rng.uniform(-1, 1, (N, D))
        if init == "sparse":                      # start velocities consistent with the sparse masks
            V = np.where(X == 1, rng.uniform(0, 1, (N, D)), rng.uniform(-6, -1, (N, D)))
        pb, pbf = X.copy(), F.copy()
        for t in range(1, T + 1):
            w = 0.9 - 0.5 * t / T
            r1, r2 = rng.random((N, D)), rng.random((N, D))
            V = np.clip(w * V + c1 * r1 * (pb - X) + c2 * r2 * (tr.best_m - X), -vmax, vmax)
            X = (rng.random((N, D)) < 1 / (1 + np.exp(-V))).astype(np.int8)
            F = np.empty(N)
            for i in range(N):
                F[i] = prob.f(X[i]); tr.offer(None, F[i], mask=X[i])
            imp = F < pbf; pb[imp] = X[imp]; pbf[imp] = F[imp]
            tr.log(t, "move", F)
    except BudgetExhausted:
        pass
    return tr.result()

def run_bgwo(prob, N, seed, init="dense", verbose=False):
    rng = np.random.default_rng(seed); tr = Tracker("BGWO", prob, verbose); D = prob.D
    T = max(2, (prob.budget - N) // N)
    try:
        X, F = _start(prob, rng, N, init, tr)
        for t in range(1, T + 1):
            a = 2 - 2 * t / T
            leaders = X[np.argsort(F)[:3]].copy()
            new = np.empty_like(X)
            for i in range(N):
                xs = [Lp - (2 * a * rng.random(D) - a) * np.abs(2 * rng.random(D) * Lp - X[i]) for Lp in leaders]
                new[i] = np.clip(np.mean(xs, axis=0), LB, UB)
            X = new
            F = np.array([prob.fx(x) for x in X])
            for x, f in zip(X, F): tr.offer(x, f)
            tr.log(t, "move", F)
    except BudgetExhausted:
        pass
    return tr.result()


# ------------------------------------------------------------------ stagnation-triggered add-on for any base optimiser
def ga_generation(rng, X, F, best_m, prob, offer, pc=0.9, tour=3):
    """One GA generation (tournament 3, uniform crossover, bit-flip 1/D, elitism of 1) on the masks of X.
    Children are stored as positions +-1 so that the shared PERTURB / RESTART operators can act on them."""
    N, D = X.shape; M = binarize(X); pm = 1.0 / D
    def pick():
        idx = rng.choice(N, tour, replace=False); return M[idx[np.argmin(F[idx])]]
    new = [best_m.copy()]
    while len(new) < N:
        p1, p2 = pick(), pick()
        if rng.random() < pc:
            cm = rng.random(D) < 0.5; kids = [np.where(cm, p1, p2), np.where(cm, p2, p1)]
        else:
            kids = [p1.copy(), p2.copy()]
        for c in kids:
            c = np.where(rng.random(D) < pm, 1 - c, c).astype(np.int8)
            if len(new) < N: new.append(c)
    X = np.where(np.array(new) == 1, 1.0, -1.0)
    F = np.empty(N)
    for i in range(N):
        F[i] = prob.fx(X[i]); offer(X[i], F[i])
    return X, F

def run_si(prob, N, seed, base="GA", init="dense", verbose=False, controller="ucb", reward="gain",
           kappa=2, tie_eps=1e-3, k_max=3, tabu_tries=20, name=None):
    """The RL-DOHO intervention layer around another optimiser ("GA" or "HO"): the base optimiser runs while
    the best subset improves; after `kappa` stalled iterations the controller picks PERTURB, RESTART or one
    base iteration, with the same gain reward, tabu memory and tie-break as RL-DOHO. Used to test whether the
    interventions help other optimisers too."""
    rng = np.random.default_rng(seed); D = prob.D; arms = ("PERTURB", "RESTART", base)
    tr = Tracker(name or f"{base}+SI", prob, verbose, tie_eps=tie_eps)
    ctrl = CONTROLLERS[controller](len(arms), seed)
    T_ho = max(1, math.ceil((prob.budget - N) / (3 * N)))
    sched = lambda T: min(T, max(1, math.ceil(T * prob.calls / prob.budget)))
    tabu = set(); g_max = 0.0; stag = 0; it = 0; last_gain = 0.0; arm_log = []
    def offer(x, f):
        tabu.add(mkey(binarize(x))); return tr.offer(x, f)
    def base_step(X, F):
        if base == "GA":
            X, F = ga_generation(rng, X, F, tr.best_m, prob, offer); return X, F, np.arange(N)
        F0 = F.copy(); ho_iteration(rng, X, F, X[np.argmin(F)].copy(), sched(T_ho), T_ho, prob, offer)
        return X, F, np.flatnonzero(F != F0)
    try:
        X = init_population(rng, N, D, init)
        F = np.array([prob.fx(x) for x in X])
        for x, f in zip(X, F): offer(x, f)
        tr.log(0, "init", F)
        while prob.left() > 0:
            it += 1
            prev_best, prev_med, calls0 = tr.best_f, float(np.median(F)), prob.calls
            before = (tr.best_f, int(tr.best_m.sum()))
            if stag < kappa:
                arm = None; X, F, moved = base_step(X, F)
            else:
                arm, how = ctrl.choose(controller_state(prob.calls, prob.budget, last_gain)); a = arms[arm]
                if a == base:
                    X, F, moved = base_step(X, F)
                else:
                    worst = np.argsort(F)[N // 2:]
                    for i in worst:
                        if a == "PERTURB":
                            def make():
                                x = tr.best_x.copy(); k = int(rng.integers(1, k_max + 1))
                                j = rng.choice(D, k, replace=False)
                                x[j] = -np.sign(x[j] + 1e-12) * rng.uniform(0.5, 2.0, k)
                                return x
                        else:
                            make = lambda: random_position(rng, D, init)
                        for _ in range(tabu_tries):
                            x = np.clip(make(), LB, UB)
                            if mkey(binarize(x)) not in tabu: break
                        X[i] = x; F[i] = prob.fx(x); offer(x, F[i])
                    moved = worst
            cost = max(1, prob.calls - calls0)
            changed = (tr.best_f, int(tr.best_m.sum())) != before
            stag = 0 if changed else stag + 1
            g = max(0.0, prev_best - tr.best_f) / cost; g_max = max(g_max, g)
            if arm is None:
                tr.log(it, "explore", F)
            else:
                share = float(np.mean(F[moved] < prev_med)) if len(moved) else 0.0
                r = (0.8 * (min(1.0, g / g_max) if g_max > 0 else 0.0) + 0.2 * share) if reward == "gain" \
                    else (1.0 if changed else 0.2 * share)
                ctrl.update(arm, r); last_gain = prev_best - tr.best_f
                arm_log.append((it, arms[arm], how, r, prev_best - tr.best_f, cost))
                tr.log(it, f"{arms[arm]}({how})", F, reward=r)
    except BudgetExhausted:
        pass
    ctrl.finish()
    res = tr.result(); res["arms"] = pd.DataFrame(arm_log, columns=["iter", "arm", "how", "reward", "gain", "cost"])
    return res

def run_ga_si(prob, N, seed, init="dense", verbose=False, **kw): return run_si(prob, N, seed, "GA", init, verbose, **kw)
def run_ho_si(prob, N, seed, init="dense", verbose=False, **kw): return run_si(prob, N, seed, "HO", init, verbose, **kw)

# ------------------------------------------------------------------ filter / embedded baselines (as in Notebooks 4 and 5)
# Each ranks features on the training data only; k is then chosen by the same fitness (CV on training data).
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif

def rank_lasso(X, y, seed, k_max):
    Xs = (X - X.mean(0)) / (X.std(0) + 1e-12)
    m = LogisticRegression(penalty="l1", solver="saga", C=0.5, max_iter=3000, random_state=seed).fit(Xs, y)
    return np.argsort(-np.abs(m.coef_).sum(0), kind="stable")

def rank_mrmr(X, y, seed, k_max):
    rel = mutual_info_classif(X, y, random_state=seed)
    Xs = (X - X.mean(0)) / (X.std(0) + 1e-12); n = len(Xs)
    sel = [int(np.argmax(rel))]; red = np.zeros(X.shape[1])
    for _ in range(min(k_max, X.shape[1], 2000) - 1):
        red += np.abs(Xs.T @ Xs[:, sel[-1]]) / n
        score = rel - red / len(sel); score[sel] = -np.inf
        sel.append(int(np.argmax(score)))
    rest = [j for j in np.argsort(-rel) if j not in set(sel)]
    return np.array(sel + rest)

def rank_relieff(X, y, seed, k_max, m=500, k=10):
    rng = np.random.default_rng(seed)
    Xn = (X - X.min(0)) / (X.max(0) - X.min(0) + 1e-12)
    classes, counts = np.unique(y, return_counts=True); prior = dict(zip(classes, counts / len(y)))
    idx = rng.choice(len(Xn), min(m, len(Xn)), replace=False); W = np.zeros(Xn.shape[1])
    for i in idx:
        d = np.abs(Xn - Xn[i]).sum(1); d[i] = np.inf
        for c in classes:
            cand = np.flatnonzero(y == c); cand = cand[cand != i]
            if len(cand) == 0: continue
            nn = cand[np.argsort(d[cand])[:k]]
            diff = np.abs(Xn[nn] - Xn[i]).mean(0)
            if c == y[i]: W -= diff / len(idx)
            else: W += prior[c] / (1 - prior[y[i]]) * diff / len(idx)
    return np.argsort(-W, kind="stable")

RANKERS = {"LASSO": rank_lasso, "mRMR": rank_mrmr, "ReliefF": rank_relieff}

def k_grid(D):
    if D <= 40: return list(range(1, D + 1))
    return sorted({k for k in [5, 10, 20, 50, 100, 200, 500, 1000, 2000, D] if k <= D})

def run_filter(name, seed, X_rank, y_rank, fitness_fn, D):
    """Returns (mask, fitness, chosen k). 'All features' keeps everything."""
    if name == "All features":
        m = np.ones(D, np.int8); return m, fitness_fn(m), D
    ks = k_grid(D); order = RANKERS[name](np.asarray(X_rank, float), np.asarray(y_rank), seed, max(ks))
    best = None
    for k in ks:
        m = np.zeros(D, np.int8); m[order[:k]] = 1
        f = fitness_fn(m)
        if best is None or f < best[1] - 1e-12: best = (m, f, k)
    return best

RUNNERS = {"DO": run_do, "HO": run_ho, "DOHO": run_doho, "RL-DOHO": run_rldoho,
           "GA": run_ga, "BPSO": run_bpso, "BGWO": run_bgwo, "GA+SI": run_ga_si, "HO+SI": run_ho_si}

print("Algorithms ready:", list(RUNNERS), "| controllers:", list(CONTROLLERS))

Algorithms ready: ['DO', 'HO', 'DOHO', 'RL-DOHO', 'GA', 'BPSO', 'BGWO', 'GA+SI', 'HO+SI'] | controllers: ['ucb', 'random', 'qlearn']


In [15]:
# ================= Statistics helpers =================
from scipy.stats import wilcoxon, friedmanchisquare, binomtest
TIE = 1e-12

def holm(ps):
    ps = np.asarray(ps, float); order = np.argsort(ps); m = len(ps); adj = np.empty(m); run = 0.0
    for i, j in enumerate(order):
        run = max(run, (m - i) * ps[j]); adj[j] = min(1.0, run)
    return adj

def wtl(a, b, higher_better):
    d = np.asarray(a) - np.asarray(b)
    w = int(np.sum(d > TIE) if higher_better else np.sum(d < -TIE)); t = int(np.sum(np.abs(d) <= TIE))
    return w, t, len(d) - w - t

def per_dataset_tests(R, ref, methods, metric, higher_better):
    """Friedman over `methods` (seeds as blocks) and Holm-corrected Wilcoxon of `ref` vs each other method."""
    rows = []
    for ds in R.dataset.unique():
        P = R[R.dataset == ds].pivot_table(index="seed", columns="method", values=metric)[methods].dropna()
        try: fp = friedmanchisquare(*[P[c] for c in methods]).pvalue
        except ValueError: fp = np.nan
        tmp = []
        for c in methods:
            if c == ref: continue
            w, t, l = wtl(P[ref], P[c], higher_better)
            try: p = wilcoxon(P[ref], P[c], zero_method="zsplit").pvalue
            except ValueError: p = 1.0
            tmp.append(dict(dataset=ds, metric=metric, vs=c, W=w, T=t, L=l, mean_diff=float((P[ref] - P[c]).mean()), p=p, friedman_p=fp))
        for r_, h in zip(tmp, holm([x["p"] for x in tmp])): r_["holm_p"] = h; rows.append(r_)
    return pd.DataFrame(rows)

def pooled_sign(R, ref, others, metric, higher_better):
    rows = []
    A = R[R.method == ref].set_index(["dataset", "seed"])[metric]
    for c in others:
        B = R[R.method == c].set_index(["dataset", "seed"])[metric]
        a, b = A.align(B, join="inner")
        w, t, l = wtl(a, b, higher_better)
        p = binomtest(w, w + l, 0.5).pvalue if w + l else 1.0
        rows.append(dict(metric=metric, vs=c, W=w, T=t, L=l, n=len(a), sign_p=p))
    out = pd.DataFrame(rows); out["holm_p"] = holm(out.sign_p.values) if len(out) else []
    return out

def mean_ranks(R, methods, metric, higher_better):
    X = R[R.method.isin(methods)].copy()
    X["r"] = X.groupby(["dataset", "seed"])[metric].rank(ascending=not higher_better)
    return X.pivot_table(index="method", columns="dataset", values="r").reindex(methods)

def summary_table(R, methods):
    g = R[R.method.isin(methods)].groupby(["dataset", "method"])
    S = g.agg(fit=("fit", "mean"), fit_sd=("fit", "std"), cv=("cv", "mean"), test_acc=("test_acc", "mean"),
              acc_sd=("test_acc", "std"), test_f1=("test_f1", "mean"), n_feats=("n_feats", "mean")).reset_index()
    return S
print("Statistics helpers ready.")

def ablation_report(A, main, variants, label):
    """A: DataFrame dataset, variant, seed, fit, ... ; compares `main` with every other variant."""
    A = A.copy(); A["r"] = A.groupby(["dataset", "seed"]).fit.rank()
    print(f"\n=== {label}: mean per-run rank on fitness (1 = best) ===")
    print(A.pivot_table(index="variant", columns="dataset", values="r").reindex(variants).round(2).assign(
        ALL=A.groupby("variant").r.mean().reindex(variants).round(2)).to_string())
    print(f"\nMean fitness / test accuracy / features:")
    print(A.groupby("variant")[["fit", "test_acc", "n_feats"]].mean().reindex(variants).round(4).to_string())
    P = A.pivot_table(index=["dataset", "seed"], columns="variant", values="fit")
    print(f"\n{main} against each variant (fitness, all runs pooled): W/T/L for {main}, two-sided sign test")
    for v in variants:
        if v == main or v not in P: continue
        a, b = P[main].dropna().align(P[v].dropna(), join="inner")
        w, t, l = wtl(a, b, False); p = binomtest(w, w + l, 0.5).pvalue if w + l else 1.0
        print(f"  vs {v:<28s} {w:>3}/{t}/{l:<3}  p = {p:.4f}")

# ================= Extra helpers for the extension notebooks =================
def record(res, prob, t0, test_fn, cache, key_fn):
    m = np.asarray(res["mask"]).astype(int); sec = time.time() - t0
    return {"fit": float(res["fit"]), "cv": float(cache[key_fn(m)]["acc"]), "n_feats": int(m.sum()), "mask": np.packbits(m),
            "n": len(m), "hist": res["hist"], "arms": res.get("arms"), "calls": prob.calls, "sec": sec,
            "fsec": prob.fsec, "overhead_sec": sec - prob.fsec, "n_unique": len(prob.seen), **test_fn(m)}

def frame(store, cols=("fit", "cv", "n_feats", "test_acc", "test_prec", "test_rec", "test_f1")):
    return pd.DataFrame([{"dataset": k[0], "method": k[1], "seed": k[2], **{c: v.get(c, np.nan) for c in cols}} for k, v in store.items()])

def sign_line(R, a, b, label, metric="fit", higher_better=False):
    P = R.pivot_table(index=["dataset", "seed"], columns="method", values=metric)
    if a not in P or b not in P: print(f"{label}: missing runs"); return None
    x, y = P[a].dropna().align(P[b].dropna(), join="inner")
    w, t, l = wtl(x, y, higher_better); p = binomtest(w, w + l, 0.5).pvalue if w + l else 1.0
    print(f"{label}: {a} W/T/L = {w}/{t}/{l} over {len(x)} paired runs (mean diff {float((x - y).mean()):+.5f}), sign test p = {p:.4f}")
    return dict(label=label, a=a, b=b, W=w, T=t, L=l, n=len(x), p=p)
print("Extension helpers ready.")

Statistics helpers ready.
Extension helpers ready.


## 6. Part A: add-ons, Q-learning and κ on the original split (resumable)

In [16]:
A_FILE, AL_FILE = f"{OUT_DIR}/v3_partA.pkl", f"{OUT_DIR}/v3_partA_long.pkl"
partA, partAL = load_pkl(A_FILE), load_pkl(AL_FILE)
VARIANTS = {"GA+SI": (run_ga_si, {}), "HO+SI": (run_ho_si, {}), "RL-DOHO [Q-learning]": (run_rldoho, dict(controller="qlearn")),
            "RL-DOHO [κ=1]": (run_rldoho, dict(kappa=1)), "RL-DOHO [κ=3]": (run_rldoho, dict(kappa=3)), "RL-DOHO [κ=4]": (run_rldoho, dict(kappa=4))}
LONG_VARIANTS = {"Q-learning (3x)": (run_rldoho, dict(controller="qlearn"))}
use_split(ORIG_SPLIT); t_start = time.time()
if RUN["part_a"]:
    for s in SEEDS:
        for v, (fn, kw) in VARIANTS.items():
            if ("Rheumatic", v, s) in partA: continue
            t0 = time.time(); prob = Problem(fitness_function, N_FEATURES, BUDGET)
            partA[("Rheumatic", v, s)] = record(fn(prob, POP_SIZE, s, init=INIT, verbose=VERBOSE, **kw), prob, t0, test_metrics, CACHES[ORIG_SPLIT], mask_str)
            save_pkl(partA, A_FILE)
        if RUN["part_a_long"]:
            for v, (fn, kw) in LONG_VARIANTS.items():
                if ("Rheumatic", v, s) in partAL: continue
                t0 = time.time(); prob = Problem(fitness_function, N_FEATURES, LONG_BUDGET)
                partAL[("Rheumatic", v, s)] = record(fn(prob, POP_SIZE, s, init=INIT, verbose=VERBOSE, **kw), prob, t0, test_metrics, CACHES[ORIG_SPLIT], mask_str)
                save_pkl(partAL, AL_FILE)
        save_cache(ORIG_SPLIT)
        print(f"\n[{time.strftime('%H:%M:%S')}] seed {s}: " + "  ".join(f"{v}={partA[('Rheumatic', v, s)]['fit']:.4f}" for v in VARIANTS if ('Rheumatic', v, s) in partA)
              + f"   elapsed {time.time() - t_start:.0f}s", flush=True)
print("Part A complete.")


[14:18:19] seed 0: GA+SI=0.1820  HO+SI=0.1839  RL-DOHO [Q-learning]=0.1884  RL-DOHO [κ=1]=0.1854  RL-DOHO [κ=3]=0.1881  RL-DOHO [κ=4]=0.1854   elapsed 0s

[14:18:19] seed 1: GA+SI=0.1825  HO+SI=0.1835  RL-DOHO [Q-learning]=0.1822  RL-DOHO [κ=1]=0.1821  RL-DOHO [κ=3]=0.1865  RL-DOHO [κ=4]=0.1884   elapsed 0s

[14:18:19] seed 2: GA+SI=0.1822  HO+SI=0.1835  RL-DOHO [Q-learning]=0.1884  RL-DOHO [κ=1]=0.1822  RL-DOHO [κ=3]=0.1821  RL-DOHO [κ=4]=0.1835   elapsed 0s

[14:18:19] seed 3: GA+SI=0.1820  HO+SI=0.1835  RL-DOHO [Q-learning]=0.1837  RL-DOHO [κ=1]=0.1835  RL-DOHO [κ=3]=0.1884  RL-DOHO [κ=4]=0.1854   elapsed 0s

[14:18:19] seed 4: GA+SI=0.1820  HO+SI=0.1837  RL-DOHO [Q-learning]=0.1851  RL-DOHO [κ=1]=0.1830  RL-DOHO [κ=3]=0.1835  RL-DOHO [κ=4]=0.1881   elapsed 1s

[14:18:19] seed 5: GA+SI=0.1825  HO+SI=0.1854  RL-DOHO [Q-learning]=0.1822  RL-DOHO [κ=1]=0.1871  RL-DOHO [κ=3]=0.1854  RL-DOHO [κ=4]=0.1881   elapsed 1s

[14:18:20] seed 6: GA+SI=0.1820  HO+SI=0.1825  RL-DOHO [Q-learning]=0

## 7. Part A results, together with the Notebook 4 and 6 runs

In [17]:
cols = ["fit", "cv", "n_feats", "test_acc", "test_prec", "test_rec", "test_f1"]
def load(p): return load_pkl(p)
v2runs, v2abl, v2long = load(f"{V2_DIR}/v2_runs.pkl"), load(f"{V2_DIR}/v2_ablation.pkl"), load(f"{V2_DIR}/v2_long.pkl")
nb4 = load(f"{PREV_DIR}/baselines_runs_p12_i20_a0.99_tie0.001.pkl")
print(f"Notebook 6: {len(v2runs)} runs, {len(v2abl)} ablation, {len(v2long)} long | Notebook 4: {len(nb4)} runs")
rows = [{"dataset": "Rheumatic", "method": k[1], "seed": k[2], **{c: v[c] for c in cols}} for k, v in partA.items()]
rows += [{"dataset": "Rheumatic", "method": k[1], "seed": k[2], **{c: v[c] for c in cols}} for k, v in v2runs.items()]
rows += [{"dataset": "Rheumatic", "method": a, "seed": s, "fit": v["fit"], "cv": v["cv_acc"], "n_feats": v["n_feats"],
          **{c: v[c] for c in ["test_acc", "test_prec", "test_rec", "test_f1"]}} for (a, s), v in nb4.items() if a in ["GA", "BPSO", "BGWO"] + FILTERS]
ABL_NAMES = {"Random arm": "RL-DOHO [random arm]", "PERTURB only": "RL-DOHO [PERTURB only]", "No intervention (DO only)": "RL-DOHO [no intervention]"}
rows += [{"dataset": "Rheumatic", "method": ABL_NAMES[k[1]], "seed": k[2], **{c: v[c] for c in cols}} for k, v in v2abl.items() if k[1] in ABL_NAMES]
RA_ = pd.DataFrame(rows); RA_ = RA_[RA_.seed.isin(SEEDS)].drop_duplicates(["method", "seed"])
LONG_NAMES = {"UCB, gain reward": "UCB1 (3x)", "Random arm": "Random arm (3x)"}
LA_ = pd.DataFrame([{"dataset": "Rheumatic", "method": k[1], "seed": k[2], **{c: v[c] for c in cols}} for k, v in partAL.items()] +
                   [{"dataset": "Rheumatic", "method": LONG_NAMES[k[1]], "seed": k[2], **{c: v[c] for c in cols}} for k, v in v2long.items() if k[1] in LONG_NAMES])
if len(LA_): LA_ = LA_[LA_.seed.isin(SEEDS)]

show = ["RL-DOHO", "GA", "GA+SI", "HO", "HO+SI", "RL-DOHO [Q-learning]", "RL-DOHO [random arm]", "RL-DOHO [κ=1]", "RL-DOHO [κ=3]", "RL-DOHO [κ=4]"]
show = [m for m in show if m in set(RA_.method)]
print(RA_[RA_.method.isin(show)].groupby("method")[["fit", "test_acc", "test_f1", "n_feats"]].agg(["mean", "std"]).reindex(show).round(4).to_string())
print()
for a, b in [("GA+SI", "GA"), ("HO+SI", "HO"), ("RL-DOHO", "GA+SI"), ("RL-DOHO [Q-learning]", "RL-DOHO [random arm]"),
             ("RL-DOHO [Q-learning]", "RL-DOHO"), ("RL-DOHO [κ=1]", "RL-DOHO"), ("RL-DOHO [κ=3]", "RL-DOHO"), ("RL-DOHO [κ=4]", "RL-DOHO")]:
    sign_line(RA_, a, b, f"  {a} vs {b}")
if len(LA_):
    print()
    for a, b in [("Q-learning (3x)", "Random arm (3x)"), ("Q-learning (3x)", "UCB1 (3x)")]: sign_line(LA_, a, b, f"  3x budget: {a} vs {b}")
    print(LA_.groupby("method")[["fit", "test_acc", "n_feats"]].mean().round(4).to_string())
best_known = min(v["fit"] for v in CACHES[ORIG_SPLIT].values())
print(f"\nBest fitness among all {len(CACHES[ORIG_SPLIT])} subsets evaluated on this split so far: {best_known:.5f}")
print("Runs reaching it:", RA_[np.isclose(RA_.fit, best_known, atol=1e-9)].groupby("method").size().to_dict())
RA_.to_csv(f"{OUT_DIR}/v3_partA_runs.csv", index=False)
if len(LA_): LA_.to_csv(f"{OUT_DIR}/v3_partA_long_runs.csv", index=False)

Notebook 6: 40 runs, 50 ablation, 30 long | Notebook 4: 110 runs
                         fit         test_acc         test_f1         n_feats        
                        mean     std     mean     std    mean     std    mean     std
method                                                                               
RL-DOHO               0.1832  0.0013   0.8433  0.0042  0.8396  0.0045    12.2  1.0328
GA                    0.1821  0.0002   0.8445  0.0035  0.8386  0.0024    10.4  0.8433
GA+SI                 0.1822  0.0003   0.8460  0.0018  0.8401  0.0018    10.1  0.3162
HO                    0.1838  0.0013   0.8429  0.0009  0.8388  0.0009    12.6  0.9661
HO+SI                 0.1839  0.0009   0.8451  0.0024  0.8415  0.0044    11.9  1.1972
RL-DOHO [Q-learning]  0.1846  0.0023   0.8442  0.0031  0.8393  0.0049    12.1  1.2867
RL-DOHO [random arm]  0.1846  0.0022   0.8437  0.0029  0.8389  0.0040    12.2  1.0328
RL-DOHO [κ=1]         0.1836  0.0017   0.8452  0.0034  0.8409  0.0040    11

## 8. Part B: every method on the new train/test splits (resumable)

In [18]:
def b_file(sp): return f"{OUT_DIR}/v3_partB_split{sp}.pkl"
def load_partB():
    out = dict(load_pkl(f"{OUT_DIR}/v3_partB.pkl"))           # older single-file version, if it exists
    for f in sorted(os.listdir(OUT_DIR)):
        if f.startswith("v3_partB_split") and f.endswith(".pkl"): out.update(load_pkl(f"{OUT_DIR}/{f}"))
    return out
partB = load_partB()
print(f"{len(partB)} Part B runs already saved")
t_start = time.time()
if RUN["part_b"]:
    for sp in EXTRA_SPLITS:
        use_split(sp); ds = f"split {sp}"
        print(f"\n=== split {sp}: cache holds {len(CACHES[sp])} subsets ===")
        S = SPLIT[sp]
        for s in SPLIT_SEEDS:
            for a in WRAPPERS + ADDONS:
                if (ds, a, s) in partB: continue
                t0 = time.time(); prob = Problem(fitness_function, N_FEATURES, BUDGET)
                partB[(ds, a, s)] = record(RUNNERS[a](prob, POP_SIZE, s, init=INIT, verbose=VERBOSE), prob, t0, test_metrics, CACHES[sp], mask_str)
                save_pkl({k: v for k, v in partB.items() if k[0] == ds}, b_file(sp))
            for a in FILTERS:
                if (ds, a, s) in partB: continue
                t0 = time.time(); m, f, k = run_filter(a, s, S["X_fit"].values, S["y_fit"], fitness_function, N_FEATURES)
                partB[(ds, a, s)] = {"fit": float(f), "cv": float(CACHES[sp][mask_str(m)]["acc"]), "n_feats": int(np.sum(m)), "k": k,
                                     "mask": np.packbits(np.asarray(m, np.uint8)), "n": N_FEATURES, "sec": time.time() - t0, **test_metrics(m)}
                save_pkl({k: v for k, v in partB.items() if k[0] == ds}, b_file(sp))
            save_cache(sp)
            print(f"\n[{time.strftime('%H:%M:%S')}] split {sp} seed {s}: " + "  ".join(f"{a}={partB[(ds, a, s)]['fit']:.4f}" for a in METHODS if (ds, a, s) in partB)
                  + f"   elapsed {time.time() - t_start:.0f}s", flush=True)
save_pkl(NEW_EVAL, f"{OUT_DIR}/v3_eval_cost_splits{'_'.join(map(str, EXTRA_SPLITS))}.pkl")
print("Part B complete for splits", EXTRA_SPLITS)

104 Part B runs already saved

=== split 1: cache holds 2800 subsets ===

[14:18:21] split 1 seed 0: RL-DOHO=0.1868  DO=0.2419  HO=0.1843  DOHO=0.1859  GA=0.1843  BPSO=0.1870  BGWO=0.1843  GA+SI=0.1858  HO+SI=0.1858  LASSO=0.1858  mRMR=0.1843  ReliefF=0.1853  All features=0.1873   elapsed 1s

[14:18:21] split 1 seed 1: RL-DOHO=0.1843  DO=0.2759  HO=0.1851  DOHO=0.1873  GA=0.1853  BPSO=0.1851  BGWO=0.1905  GA+SI=0.1843  HO+SI=0.1849  LASSO=0.1858  mRMR=0.1843  ReliefF=0.1853  All features=0.1873   elapsed 1s

[14:18:22] split 1 seed 2: RL-DOHO=0.1843  DO=0.1976  HO=0.1851  DOHO=0.1859  GA=0.1853  BPSO=0.1859  BGWO=0.1868  GA+SI=0.1858  HO+SI=0.1851  LASSO=0.1858  mRMR=0.1870  ReliefF=0.1858  All features=0.1873   elapsed 1s

[14:18:22] split 1 seed 3: RL-DOHO=0.1859  DO=0.2196  HO=0.1851  DOHO=0.1859  GA=0.1853  BPSO=0.1843  BGWO=0.1907  GA+SI=0.1849  HO+SI=0.1851  LASSO=0.1858  mRMR=0.1849  ReliefF=0.1853  All features=0.1873   elapsed 1s

[14:18:22] split 1 seed 4: RL-DOHO=0.1849  DO=

## 9. Part B results: 5 splits × 5 seeds (original split = seeds 0–4 of Notebooks 4 and 6 plus Part A)

In [19]:
pd.set_option("display.width", 220)
partB = load_partB()                                           # picks up splits finished in other sessions too
rows = [{"dataset": k[0], "method": k[1], "seed": k[2], **{c: v[c] for c in cols}} for k, v in partB.items()]
orig = RA_[RA_.method.isin(METHODS) & RA_.seed.isin(SPLIT_SEEDS)].assign(dataset="split 42 (original)")
RB = pd.concat([orig, pd.DataFrame(rows)], ignore_index=True)
MB = [m for m in METHODS if m in set(RB.method)]
print(RB.groupby(["dataset", "method"]).size().unstack(0).reindex(MB).to_string())

SB = summary_table(RB, MB)
print("\nMean fitness per split:"); print(SB.pivot_table(index="method", columns="dataset", values="fit").reindex(MB).round(4).to_string())
print("\nMean test accuracy per split:"); print(SB.pivot_table(index="method", columns="dataset", values="test_acc").reindex(MB).round(4).to_string())
rk = mean_ranks(RB, MB, "fit", False); ra = mean_ranks(RB, MB, "test_acc", True)
print("\nMean per-run rank on fitness (1 = best), per split and overall:"); print(rk.assign(ALL=rk.mean(1)).round(2).to_string())
print("\nMean per-run rank on test accuracy:"); print(ra.assign(ALL=ra.mean(1)).round(2).to_string())

# Friedman with (split, seed) as blocks, and pooled sign tests RL-DOHO vs each (H8)
from scipy.stats import friedmanchisquare
for m, hb in [("fit", False), ("test_acc", True), ("test_f1", True)]:
    P = RB.pivot_table(index=["dataset", "seed"], columns="method", values=m)[MB].dropna()
    print(f"\nFriedman over {len(MB)} methods, {len(P)} blocks ({m}): p = {friedmanchisquare(*[P[c] for c in MB]).pvalue:.3g}")
POOLB = pd.concat([pooled_sign(RB, "RL-DOHO", [m for m in MB if m != "RL-DOHO"], m, hb) for m, hb in [("fit", False), ("test_acc", True), ("test_f1", True)]])
print("\n=== H8: RL-DOHO W/T/L over all splits x seeds, two-sided sign test, Holm-corrected per metric ===")
print(POOLB.round(4).to_string(index=False))
print("\nWithin the DO/HO family, per split:")
fr = mean_ranks(RB, FAMILY, "fit", False); print(fr.assign(ALL=fr.mean(1)).round(2).to_string())
print("\nAdd-ons across splits:")
for a, b in [("GA+SI", "GA"), ("HO+SI", "HO"), ("RL-DOHO", "GA+SI")]: sign_line(RB, a, b, f"  {a} vs {b}")
RB.to_csv(f"{OUT_DIR}/v3_partB_runs.csv", index=False); POOLB.to_csv(f"{OUT_DIR}/v3_partB_pooled.csv", index=False)

dataset       split 1  split 2  split 42 (original)
method                                             
RL-DOHO             5        5                    5
DO                  5        5                    5
HO                  5        5                    5
DOHO                5        5                    5
GA                  5        5                    5
BPSO                5        5                    5
BGWO                5        5                    5
GA+SI               5        5                    5
HO+SI               5        5                    5
LASSO               5        5                    5
mRMR                5        5                    5
ReliefF             5        5                    5
All features        5        5                    5

Mean fitness per split:
dataset       split 1  split 2  split 42 (original)
method                                             
RL-DOHO        0.1852   0.1820               0.1837
DO             0.2303   0.2325         

## 10. Cost on the clinical data (Part B runs; every method starts with an empty cache on each new split)

In [20]:
TM = WRAPPERS + ADDONS
TT = pd.DataFrame([{"dataset": k[0], "method": k[1], "seed": k[2], "overhead_sec": v["overhead_sec"], "n_unique": v["n_unique"]}
                   for k, v in partB.items() if "overhead_sec" in v])
if len(TT):
    cost = {f"split {sp}": c[1] / max(c[0], 1) for sp, c in NEW_EVAL.items()}
    print("Mean cost of one new fitness evaluation (s):", {k: round(v, 3) for k, v in cost.items()})
    TT["cold_sec"] = TT.overhead_sec + TT.n_unique * TT.dataset.map(cost)
    print(TT.groupby("method")[["overhead_sec", "n_unique", "cold_sec"]].mean().reindex([m for m in TM if m in set(TT.method)]).round(2).to_string())
    TT.to_csv(f"{OUT_DIR}/v3_timing.csv", index=False)

Mean cost of one new fitness evaluation (s): {'split 2': 3.643}
         overhead_sec  n_unique  cold_sec
method                                   
RL-DOHO          0.03      82.5    299.48
DO               0.01      12.6     45.91
HO               0.03     130.5    483.09
DOHO             0.02      76.6    278.34
GA               0.02     132.0    485.99
BPSO             0.02     244.5    888.18
BGWO             0.03      53.5    199.66
GA+SI            0.03     170.5    625.90
HO+SI            0.03     140.0    518.06


## 11. Summary
Per-run results: `v3_partA.pkl`, `v3_partA_long.pkl`, `v3_partB.pkl` and the CSV files. All files are in `OUT_DIR`.

In [21]:
# Zip the results of Notebooks 8 and 9 (no data files or caches) and download the zip
import os, zipfile
from google.colab import files

ROOT = '/content/drive/MyDrive'
FOLDERS = ['rl_doho_results_v3_knn', 'rl_doho_results_v3_rheum']
SKIP = ('cache_', 'fitness_cache_', 'test_cache_')          # large and can be rebuilt, so left out
OUT = '/content/rl_doho_v3_results.zip'

with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in FOLDERS:
        base = f'{ROOT}/{folder}'
        if not os.path.exists(base):
            print('not found:', base); continue
        for dirpath, dirnames, filenames in os.walk(base):
            if os.path.basename(dirpath) == 'data': continue
            for f in filenames:
                if f.startswith(SKIP) or f.endswith('.tmp'): continue
                p = os.path.join(dirpath, f)
                z.write(p, os.path.relpath(p, ROOT))
                print('added', os.path.relpath(p, ROOT))

print(f'\nZip size: {os.path.getsize(OUT) / 1e6:.1f} MB')
files.download(OUT)

added rl_doho_results_v3_knn/v3_runs.pkl
added rl_doho_results_v3_knn/v3_eval_cost.pkl
added rl_doho_results_v3_knn/v3_all_runs.csv
added rl_doho_results_v3_knn/v3_long_runs.csv
added rl_doho_results_v3_knn/v3_long.pkl
added rl_doho_results_v3_knn/v3_tab_stats_per_dataset.csv
added rl_doho_results_v3_knn/v3_tab_stats_pooled.csv
added rl_doho_results_v3_knn/v3_tab_summary.csv
added rl_doho_results_v3_knn/v3_timing.csv
added rl_doho_results_v3_knn/v3_overhead_benchmark.csv
added rl_doho_results_v3_rheum/v3_partA.pkl
added rl_doho_results_v3_rheum/v3_partA_long.pkl
added rl_doho_results_v3_rheum/v3_partA_long_runs.csv
added rl_doho_results_v3_rheum/v3_partA_runs.csv
added rl_doho_results_v3_rheum/v3_partB.pkl
added rl_doho_results_v3_rheum/v3_partB_split1.pkl
added rl_doho_results_v3_rheum/v3_partB_split2.pkl
added rl_doho_results_v3_rheum/v3_eval_cost_splits1_2.pkl
added rl_doho_results_v3_rheum/v3_partB_runs.csv
added rl_doho_results_v3_rheum/v3_partB_pooled.csv
added rl_doho_results_v3

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>